# AURA Motor Model — Research Notebooks

These notebooks support the motor-skills modeling pipeline:

- **Dataset**: one row per session (your final `motor_sessions.csv`)
- **Goal**: train a *hostable* model that outputs an `impairment_score` in **[0,1]**
  - **0** = better motor performance in this task
  - **1** = more assistance needed in this task
- **Leakage control**: splits are grouped by **userId** (no user appears in train and test)

> ⚠️ This is a functional interaction score for this game task, **not a medical diagnosis**.


## 03 — Error analysis

Explores where predictions differ from the PCA-derived target and whether performance quality signals explain error.

In [ ]:
import os, pandas as pd, numpy as np

OUTDIR = os.environ.get("AURA_OUTDIR", "../model_registry/motor/1.0.0")
df = pd.read_csv(os.path.join(OUTDIR, "reports", "sessions_with_scores.csv"))
df["abs_err_B2"] = np.abs(df["impairment_score_B2_pred"] - df["impairment_score_A"])

df.sort_values("abs_err_B2", ascending=False).head(10)[
    ["sessionId","userId","impairment_score_A","impairment_score_B2_pred","abs_err_B2"]
]


In [ ]:
perf_cols = [c for c in [
    "perf_samplingHzEstimated","perf_avgFrameMs","perf_p95FrameMs","perf_droppedFrames","perf_inputLagMsEstimate"
] if c in df.columns]

corrs = {}
for c in perf_cols:
    corrs[c] = float(np.corrcoef(df["abs_err_B2"].values, df[c].fillna(df[c].median()).values)[0,1])
corrs


In [ ]:
import matplotlib.pyplot as plt

for c in perf_cols:
    plt.figure()
    plt.scatter(df[c].values, df["abs_err_B2"].values, s=10)
    plt.xlabel(c)
    plt.ylabel("abs error (B2 vs A)")
    plt.title(f"Error vs {c}")
    plt.show()


## Notes for reporting

If errors increase with input lag/dropped frames, you can justify a data-quality control threshold in your methodology.
